# logit-thermo A2-17：文章の種類（論文・コード・小説）を変えて、A2-16 と同じ場の生成（q つき）——ランタイム 2/3（GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。
- **3 本のノートブック（ランタイム 1/3・2/3・3/3）を、別々のランタイムで同時に回してください。** 52 個の仕事（3 種類 × 4 モデル × h ＝ −0.3・0・＋0.3 の 36 個と、4 種類 × 4 モデルの「人の続きを読ませる」短い仕事 16 個）を、3 つのランタイムが「まだ終わっていない・誰も取っていない」ものから取りに行きます。3 つ同時なら 7〜8 時間くらいの見込みです。1 つだけでも、時間はかかりますが全部回せます。
- 文章：arXiv の論文（2025-04・05、40 本）、GitHub のコード（A2-3 と同じ 35 本。A2-3 の目録と一致するか確かめます）、グーテンベルクの小説（本の番号 30000 より後、本文の 30% の位置から、40 本）。種で決めて選びます（3 つのランタイムで同じになることを確かめます）。最初に文章を全部読み込み、失敗したらそこで止まって ⚠ を送ります。
- 作り：A2-16（G-pre）と同じ（温度 0.4〜0.7 の 7 個、サンプル 4、512 歩、q つき）。モデル：Pythia-1.4B・SmolLM2-1.7B・Qwen2.5-1.5B・BLOOM-1.1B。人の書いた続きを読ませたときの −log p も、短い仕事で記録します（覚えているかの記録）。
- **データは、ゲートを凍結するまで開けません。** 結果も文章の中身も表示しません。
- 計算のコード：A2-2・A2-4・A2-7・A2-7b のデータ集めのコード、`scripts/a2_explore_field.py`・`a2_explore_dsc.py`・`a2_explore_dsc2.py`（`cddae78690360418…`）、`scripts/a2_gpre_collect.py`（`49aa63e0c3479f98…`）は Drive の前のノートブックから取り出してハッシュで確かめる。`scripts/a2_3_collect.py`（`65b16cb5baccaa19…`、コードの選び方）は A2-3 のノートブックから取り出してハッシュで確かめる。`scripts/a2_17_collect.py`（`c3091b6ad3019fa7…`）は、Drive の `A2_explore/a2_17_collect.py` から読んでハッシュで確かめる。
- 保存先 `A2_17/`（種類ごとのフォルダ）。途中で切れても、もう一度「すべて実行」すれば、終わった仕事は飛ばします（取りかけの仕事は 2 時間たつとほかのランタイムが取り直せます）。
- **通知**：🚀・仕事ごとの ✅・🎉（このランタイムで回した数と、まだ残っている仕事）・⚠️。名札は「logit-thermo A2-17 文章の種類の確かめ（論文・コード・小説 × 4 モデル × 場 ±0.3、ランタイム 2/3、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-17 文章の種類の確かめ（論文・コード・小説 × 4 モデル × 場 ±0.3、ランタイム 2/3、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-17 genre collect worker 2of3', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRC24': '6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57', 'SRC7C': '8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550', 'SRC7BC': 'b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0', 'SRCF': '0ac42c7ab82b06d2e7443ba9c0c0e147c0121b5351e0b5830b1a54b46043190d', 'SRCD': '33f47a0a3bbda18341205dedb94c6d200f295fc4136f5cb8e0ee067b2d7f518c', 'SRCD2': 'cddae786903604183e46708a6606b482f720bba0b0df0a8df724440c9bde51b2', 'SRCG': '49aa63e0c3479f9841bf4b17740325fe21ddaea8b118f03730461959ef03e44b', 'SRC23': '65b16cb5baccaa19855f1d23ebb666baaac7c69c9732704307bd297658e6bbc9'})
SRC22, SRC24, SRC7C, SRC7BC, SRCF, SRCD = _s['SRC22'], _s['SRC24'], _s['SRC7C'], _s['SRC7BC'], _s['SRCF'], _s['SRCD']
SRCD, SRCD2 = _s['SRCD'], _s['SRCD2']   # 名前を 2 つ並べて置く（take_src が読み飛ばす形）
SRCD2, SRCG = _s['SRCD2'], _s['SRCG']   # 名前を 2 つ並べて置く（take_src が読み飛ばす形）
SRCG, SRC23 = _s['SRCG'], _s['SRC23']   # 名前を 2 つ並べて置く（take_src が読み飛ばす形）
def take_file(name, want):
    """Drive の logit-thermo のフォルダから、名前とハッシュが合う .py のファイルを読む。"""
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/' + name, recursive=True)):
        s = open(p, encoding='utf-8').read()
        if hashlib.sha256(s.encode()).hexdigest() == want:
            return s
    raise AssertionError(('Drive に ' + name + ' （ハッシュが合うもの）が見つかりません', want))
SRC17 = take_file('a2_17_collect.py', 'c3091b6ad3019fa74edeef05b483299dab0beb6a7676681b0b602fe5562735b9')
exec(SRC22)
exec(SRC24)
exec(SRC7C)
exec(SRC7BC)
exec(SRCF)
exec(SRCD)
exec(SRCD2)
exec(SRCG)
exec(SRC17)
# A2-3 のコードの選び方は、名前がぶつからないように別の名前空間で読み込み、関数だけを渡す
_ns3 = dict(globals()); exec(SRC23, _ns3); A3_LOAD_CODE = _ns3['load_code_docs']
print('コードを置きました')


In [ ]:
# セル3：自己テスト → BBC の記事（A2-2 と同じか確かめる）→ 仕事を取りに行って回す → 保存（結果も文章も表示しない）
import json, sys, transformers
try:
    print('自己テスト：', self_test_a27('cuda'), self_test_7b('cuda'), self_test_field('cuda'), self_test_dsc('cuda'), self_test_dsc2('cuda'), self_test_gpre(), self_test_a17('cuda'))
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_17', exist_ok=True)
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    D = prepare_docs('/content/drive/MyDrive/logit-thermo/A2_17', docs, '/content/drive/MyDrive/logit-thermo' + '/A2_3/code_meta.json', notify=print)   # 文章の読み込み（失敗したら、ここで止まって ⚠）
    N_DONE, LEFT = run_a17(1, 3, D, '/content/drive/MyDrive/logit-thermo/A2_17', 'cuda', notify=lambda s: notify(s, tags='warning') if s.startswith('⚠') else notify('✅ ' + s, tags='white_check_mark'))
    json.dump(dict(worker=1, done=N_DONE, left=LEFT, models=A17_MODELS, genres=A17_GENRES, h=GP_H, temps=GP_TEMPS, samples=GP_S, chunk=GP_CHUNK, seed=GP_SEED,
                   a17_sha256='c3091b6ad3019fa74edeef05b483299dab0beb6a7676681b0b602fe5562735b9', a23_sha256='65b16cb5baccaa19855f1d23ebb666baaac7c69c9732704307bd297658e6bbc9', gpre_sha256='49aa63e0c3479f9841bf4b17740325fe21ddaea8b118f03730461959ef03e44b', dsc2_sha256='cddae786903604183e46708a6606b482f720bba0b0df0a8df724440c9bde51b2', torch=torch.__version__, transformers=transformers.__version__,
                   python=sys.version.split()[0], gpu=torch.cuda.get_device_name(0), matmul=torch.get_float32_matmul_precision()),
              open('/content/drive/MyDrive/logit-thermo/A2_17' + '/run_meta_worker1.json', 'w'), ensure_ascii=False, indent=1)
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-17 文章の種類の確かめ（ランタイム 2/3）で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify(f'🎉 終わりました：A2-17 文章の種類の確かめ（ランタイム 2/3）。このランタイムで {N_DONE} 個。まだ残っている仕事：{len(LEFT)} 個' + (f'（{", ".join(LEFT)}）' if LEFT else '（全部そろいました）'), tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
